In [7]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/frontal_cortex")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/code/enrichment_fxns.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




Here I perform enrichment analysis to find modules enriched for cell type markers. 

These modules will later be used to correlate with exon PSI to define cell type-specific exons.

In [13]:
mod_def <- "PosBC"

# Prep marker genes

## MO marker genes

In [5]:
load("/mnt/lareaulab/reliscu/data/gene_sets/Oldham/MO_sets.RData")

In [6]:
sets <- c(
    "BARRES_ASTROCYTES",
    "BARRES_OLIGODENDROCYTES",
    "HICKMAN_MICROGLIA_SENSOME_99",
    "BARRES_NEURONS",
    "BUTLER_ENDOTHELIAL_HPA",
    "ZEISEL_MURAL",
    "ZEISEL_EPENDYMAL",
    "ZEISEL_INTERNEURON",
    "ZHANG_MOUSE_OPC_2014",
    "Mukamel_InhibitoryNeurons",
    "Mukamel_ExcitatoryNeurons",
    "SUGINO_UP_GABAERGIC_NEURONS",
    "SUGINO_UP_GLUTAMATERGIC_NEURONS",
    "SUGINO_UP_LAYERS4-6_GABAERGIC_NEURONS_CINGULATE_CTX",
    "BAKKEN_2019_PVALB_GABAERGIC_DE_GABA_CLUSTERS",
    "BAKKEN_2019_VIP_GABAERGIC_DE_GABA_CLUSTERS",
    "BAKKEN_2019_SST_GABAERGIC_DE_GABA_CLUSTERS",
    "BAKKEN_2019_LAMP5_GABAERGIC_DE_GABA_CLUSTERS",
    "BAKKEN_2019_SNCG_GABAERGIC_DE_GABA_CLUSTERS",
    "BAKKEN_2019_SST_CHODL_GABAERGIC_DE_GABA_CLUSTERS",
    "VELMESHEV_2019_IN_SST",
    MO_legend$SetName[grepl("schirmer", MO_legend$SetName, ignore.case=T)],
    MO_legend$SetName[grepl("yang_pfc", MO_legend$SetName, ignore.case=T)]
)

In [7]:
mask <- MO_legend$SetName %in% sets

marker_genes_list <- MO_sets[mask]
names(marker_genes_list) <- MO_legend$SetName[mask] 

marker_genes_list <- lapply(marker_genes_list, toupper)

## Gugene's marker gene

In [9]:
marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/kang_2026_preprint/kang_2026_preprint_DE_genes.RData")

# Get enrichments

In [10]:
set_source <- "kang_2026"

In [ ]:
network_dir <- "GTEx_cortex_TPM_rd2_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_Modules"

In [ ]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [11]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

In [12]:
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
YANG_PFC_2021_OLIGODENDROCYTE,1.044100e-277,4.872918e-273,pink,Bicor-None_signum0.284_minSize10_merge_ME_0.9_18656
YANG_PFC_2021_ASTROCYTE,8.193355e-254,6.373201e-250,brown,Bicor-None_signum0.452_minSize6_merge_ME_0.9_18656
YANG_PFC_2021_ENDOTHELIAL,4.365030e-153,1.072212e-149,brown,Bicor-None_signum0.284_minSize12_merge_ME_0.9_18656
YANG_PFC_2021_MICROGLIA,5.407079e-129,6.820373e-126,turquoise,Bicor-None_signum0.347_minSize10_merge_ME_0.9_18656
YANG_PFC_2021_NRGN_NEURON,1.649149e-125,1.973523e-122,blue,Bicor-None_signum0.284_minSize12_merge_ME_0.9_18656
YANG_PFC_2021_L4_EXCITATORY_NEURON,1.069911e-83,7.343210e-81,turquoise,Bicor-None_signum0.452_minSize6_merge_ME_0.9_18656
BUTLER_ENDOTHELIAL_HPA,6.480921e-78,4.032948e-75,greenyellow,Bicor-None_signum0.347_minSize8_merge_ME_0.9_18656
YANG_PFC_2021_PV_INTERNEURON,4.763243e-74,2.925070e-71,grey60,Bicor-None_signum0.284_minSize8_merge_ME_0.9_18656
YANG_PFC_2021_L5_6_EXCITATORY_NEURON,8.855869e-61,3.936307e-58,blue,Bicor-None_signum0.347_minSize6_merge_ME_0.9_18656


In [ ]:
write.csv(top_mods_df, file=paste0("data/enrichments/_", network_dir, "_", set_source, "_Qval_modules.csv"), row.names=FALSE, quote=FALSE)